In [2]:
import os
import glob
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')

print("="*80)
print("CAMINHO A: RECONSTRUÇÃO TOTAL COM GABARITO DINÂMICO E EXPORTAÇÃO")
print("="*80)

# ---------------------------------------------------------
# 1. PARSER DO AOI.CSV (Extração das Coordenadas)
# ---------------------------------------------------------
print("Lendo e decodificando o aoi.csv...")
caminho_aoi = '/workspaces/EyeTracking/data/aoi/aoi.csv'
df_aoi = pd.read_csv(caminho_aoi)

dicionario_aoi = {}

def extrair_xy(texto):
    if pd.isna(texto) or not isinstance(texto, str): return None
    match = re.search(r'\((\d+),\s*(\d+)\)', texto)
    if match: return int(match.group(1)), int(match.group(2))
    return None

for col in df_aoi.columns[1:]:
    try:
        face_id = int(col)
        dicionario_aoi[face_id] = {}
        for i, regiao in enumerate(['Olho_D', 'Olho_E', 'Nariz', 'Boca']):
            p1 = extrair_xy(df_aoi.loc[i*2, col])
            p2 = extrair_xy(df_aoi.loc[(i*2)+1, col])
            if p1 and p2:
                dicionario_aoi[face_id][regiao] = (min(p1[0], p2[0]), min(p1[1], p2[1]), max(p1[0], p2[0]), max(p1[1], p2[1]))
    except:
        continue

# ---------------------------------------------------------
# 2. REPROCESSAMENTO E EXPLOSÃO DE FEATURES
# ---------------------------------------------------------
print("Aplicando AOIs dinâmicos e explodindo variáveis...")
PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
PASTA_PROCESSED = '/workspaces/EyeTracking/data/processed/'
arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_pacientes = []

def dentro_da_caixa(x, y, caixa):
    if pd.isna(x) or pd.isna(y) or caixa is None: return False
    return (caixa[0] <= x <= caixa[2]) and (caixa[1] <= y <= caixa[3])

for caminho in arquivos:
    nome = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome.upper() else "CONTROLE"
    
    try:
        df = pd.read_csv(caminho)
        df = df[df['Fase_Estimulo'] == 'Exposicao_Face'].copy()
        if len(df) == 0: continue
            
        def checar_aoi(row, regiao):
            f_id = row['Face_ID']
            if f_id in dicionario_aoi and regiao in dicionario_aoi[f_id]:
                return dentro_da_caixa(row['Gaze_X'], row['Gaze_Y'], dicionario_aoi[f_id][regiao])
            return False

        df['In_Olho_D'] = df.apply(lambda r: checar_aoi(r, 'Olho_D'), axis=1)
        df['In_Olho_E'] = df.apply(lambda r: checar_aoi(r, 'Olho_E'), axis=1)
        df['In_Nariz'] = df.apply(lambda r: checar_aoi(r, 'Nariz'), axis=1)
        df['In_Boca'] = df.apply(lambda r: checar_aoi(r, 'Boca'), axis=1)
        df['In_Olhos_Total'] = df['In_Olho_D'] | df['In_Olho_E']
        df['Pupila_Media'] = df[['Pupil_L', 'Pupil_R']].mean(axis=1)
        
        features_paciente = {'Paciente': nome, 'Grupo': grupo}
        emocoes = ['Raiva', 'Feliz', 'Neutro']
        
        for emocao in emocoes:
            df_emocao = df[(df['Tipo_Estimulo'] == 'Humano') & (df['Emocao'] == emocao)]
            if len(df_emocao) == 0: continue
                
            features_paciente[f'Perc_Olhos_{emocao}'] = df_emocao['In_Olhos_Total'].mean() * 100
            features_paciente[f'Perc_Boca_{emocao}'] = df_emocao['In_Boca'].mean() * 100
            features_paciente[f'Perc_Nariz_{emocao}'] = df_emocao['In_Nariz'].mean() * 100
            features_paciente[f'Pupila_Media_{emocao}'] = df_emocao['Pupila_Media'].mean()
            features_paciente[f'Pupila_Std_{emocao}'] = df_emocao['Pupila_Media'].std()
            features_paciente[f'GazeX_Std_{emocao}'] = df_emocao['Gaze_X'].std()
            features_paciente[f'GazeY_Std_{emocao}'] = df_emocao['Gaze_Y'].std()

        lista_pacientes.append(features_paciente)
    except Exception as e:
        print(f"Erro no paciente {nome}: {e}")

df_final = pd.DataFrame(lista_pacientes).fillna(0)

# Criando os Deltas
for col in ['Perc_Olhos', 'Perc_Boca', 'Perc_Nariz', 'Pupila_Media', 'GazeX_Std']:
    if f'{col}_Neutro' in df_final.columns and f'{col}_Raiva' in df_final.columns:
        df_final[f'Delta_NeutroRaiva_{col}'] = df_final[f'{col}_Neutro'] - df_final[f'{col}_Raiva']
    if f'{col}_Feliz' in df_final.columns and f'{col}_Raiva' in df_final.columns:
        df_final[f'Delta_FelizRaiva_{col}'] = df_final[f'{col}_Feliz'] - df_final[f'{col}_Raiva']

# --- SALVANDO A NOVA MATRIZ ---
CAMINHO_SALVAR = os.path.join(PASTA_PROCESSED, 'matriz_features_dinamicas.csv')
df_final.to_csv(CAMINHO_SALVAR, index=False)
print(f"✅ Nova matriz salva com sucesso em: {CAMINHO_SALVAR}")
print(f"Total de Features Extraídas: {df_final.shape[1] - 2}")

# ---------------------------------------------------------
# 3. AUDITORIA: CAÇADA AOS 89% (O Teste de Stress)
# ---------------------------------------------------------
y = (df_final['Grupo'] == 'TEA').astype(int)
X = df_final.drop(columns=['Paciente', 'Grupo'])
X_scaled = StandardScaler().fit_transform(X)

print("\nRodando a roleta da semente (Seed Hunting) no Gradient Boosting...\n")

melhor_acc_split = 0
semente_split = 0
resultados_split = []

for seed in range(1, 101):
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=seed, stratify=y)
    clf = GradientBoostingClassifier(random_state=42) 
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    resultados_split.append(acc)
    if acc > melhor_acc_split:
        melhor_acc_split = acc
        semente_split = seed

melhor_acc_loocv = 0
loo = LeaveOneOut()

for seed in range(1, 101):
    clf_loo = GradientBoostingClassifier(random_state=seed)
    previsoes = []
    for train_idx, test_idx in loo.split(X_scaled):
        clf_loo.fit(X_scaled[train_idx], y.iloc[train_idx])
        previsoes.append(clf_loo.predict(X_scaled[test_idx])[0])
    acc = accuracy_score(y, previsoes)
    if acc > melhor_acc_loocv:
        melhor_acc_loocv = acc

clear_output(wait=True)
print("="*80)
print(f"{'RESULTADO DA AUDITORIA: A ORIGEM DOS 89%':^80}")
print("="*80)
print(f"Total de Features Reais Processadas: {X.shape[1]}")
print("-" * 80)
print(f"CENÁRIO 1: O Falso Positivo Metodológico (Split 80/20)")
print(f"  Acurácia Máxima: {melhor_acc_split * 100:.1f}% (Encontrada na Semente {semente_split})")
print(f"  Acurácia Mínima: {min(resultados_split) * 100:.1f}%")
print(f"  Variação: {(melhor_acc_split - min(resultados_split)) * 100:.1f} pontos percentuais por mudança de semente.")
print("-" * 80)
print(f"CENÁRIO 2: O Teste de Realidade (LOOCV Blindado)")
print(f"  Acurácia Máxima Possível: {melhor_acc_loocv * 100:.1f}%")
print("="*80)
print(f"Arquivo pronto para o Motor Optuna: matriz_features_dinamicas.csv")

                    RESULTADO DA AUDITORIA: A ORIGEM DOS 89%                    
Total de Features Reais Processadas: 31
--------------------------------------------------------------------------------
CENÁRIO 1: O Falso Positivo Metodológico (Split 80/20)
  Acurácia Máxima: 100.0% (Encontrada na Semente 89)
  Acurácia Mínima: 25.0%
  Variação: 75.0 pontos percentuais por mudança de semente.
--------------------------------------------------------------------------------
CENÁRIO 2: O Teste de Realidade (LOOCV Blindado)
  Acurácia Máxima Possível: 65.8%
Arquivo pronto para o Motor Optuna: matriz_features_dinamicas.csv


In [7]:
import os
import glob
import pandas as pd
import numpy as np
import re
import optuna
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
import xgboost as xgb
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("="*80)
print("CAMINHO B: MOTOR OPTUNA + SCANPATH DINÂMICO (AOI.CSV)")
print("="*80)

# ---------------------------------------------------------
# 1. PARSER DO AOI.CSV E EXTRAÇÃO DO MARKOV DINÂMICO
# ---------------------------------------------------------
print("Calculando Matriz de Transição de Markov com base nas marcações manuais...")
caminho_aoi = '/workspaces/EyeTracking/data/aoi/aoi.csv'
df_aoi = pd.read_csv(caminho_aoi)
dicionario_aoi = {}

def extrair_xy(texto):
    if pd.isna(texto) or not isinstance(texto, str): return None
    match = re.search(r'\((\d+),\s*(\d+)\)', texto)
    return (int(match.group(1)), int(match.group(2))) if match else None

for col in df_aoi.columns[1:]:
    try:
        f_id = int(col)
        dicionario_aoi[f_id] = {}
        for i, regiao in enumerate(['Olho_D', 'Olho_E', 'Nariz', 'Boca']):
            p1, p2 = extrair_xy(df_aoi.loc[i*2, col]), extrair_xy(df_aoi.loc[(i*2)+1, col])
            if p1 and p2:
                dicionario_aoi[f_id][regiao] = (min(p1[0], p2[0]), min(p1[1], p2[1]), max(p1[0], p2[0]), max(p1[1], p2[1]))
    except: pass

def dentro_caixa(x, y, caixa):
    if pd.isna(x) or pd.isna(y) or not caixa: return False
    return (caixa[0] <= x <= caixa[2]) and (caixa[1] <= y <= caixa[3])

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_markov = []
estados = ['Olhos', 'Nariz', 'Boca', 'Fora']

for caminho in arquivos:
    nome = os.path.basename(caminho).replace('.csv', '')
    try:
        df = pd.read_csv(caminho)
        df_raiva = df[(df['Fase_Estimulo'] == 'Exposicao_Face') & (df['Tipo_Estimulo'] == 'Humano') & (df['Emocao'] == 'Raiva')].copy()
        
        if len(df_raiva) == 0: continue
            
        def classificar_estado(row):
            fid = row['Face_ID']
            if fid not in dicionario_aoi: return 'Fora'
            x, y = row['Gaze_X'], row['Gaze_Y']
            if dentro_caixa(x, y, dicionario_aoi[fid].get('Olho_D')) or dentro_caixa(x, y, dicionario_aoi[fid].get('Olho_E')): return 'Olhos'
            if dentro_caixa(x, y, dicionario_aoi[fid].get('Boca')): return 'Boca'
            if dentro_caixa(x, y, dicionario_aoi[fid].get('Nariz')): return 'Nariz'
            return 'Fora'

        df_raiva['Estado_Atual'] = df_raiva.apply(classificar_estado, axis=1)
        
        # Colapso do Scanpath (Remover estados repetidos em sequência)
        mudancas = df_raiva[df_raiva['Estado_Atual'] != df_raiva['Estado_Atual'].shift(1)]
        sequencia = mudancas['Estado_Atual'].tolist()
        
        features_paciente = {'Paciente': nome}
        for o in estados:
            for d in estados: features_paciente[f"Trans_{o}_{d}"] = 0.0
                
        transicoes = pd.DataFrame({'Origem': sequencia[:-1], 'Destino': sequencia[1:]})
        if len(transicoes) > 0:
            matriz_prob = pd.crosstab(transicoes['Origem'], transicoes['Destino'], normalize='index')
            for o in matriz_prob.index:
                for d in matriz_prob.columns: features_paciente[f"Trans_{o}_{d}"] = matriz_prob.loc[o, d] * 100
                    
        lista_markov.append(features_paciente)
    except: pass

df_markov = pd.DataFrame(lista_markov)

# ---------------------------------------------------------
# 2. FUSÃO DE MATRIZES
# ---------------------------------------------------------
CAMINHO_PROCESSED = '/workspaces/EyeTracking/data/processed/matriz_features_dinamicas.csv'
df_features = pd.read_csv(CAMINHO_PROCESSED)
df_fusion = pd.merge(df_features, df_markov, on='Paciente', how='inner')

y = (df_fusion['Grupo'] == 'TEA').astype(int)
X = df_fusion.drop(columns=['Paciente', 'Grupo'])
X = X.loc[:, (X != 0).any(axis=0)] # Remove colunas sem variação

print(f"Fusão Concluída. Iniciando Optuna com a Super Matriz de {X.shape[1]} features...")

# ---------------------------------------------------------
# 3. O MOTOR OPTUNA
# ---------------------------------------------------------
def calcular_score_clinico(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        vp, fn, vn, fp = cm[1][1], cm[1][0], cm[0][0], cm[0][1]
        sens = vp / (vp + fn) if (vp + fn) > 0 else 0
        esp = vn / (vn + fp) if (vn + fp) > 0 else 0
    else: sens, esp = 0, 0
    return (0.45 * sens) + (0.30 * acc) + (0.15 * f1) + (0.10 * esp), acc, sens, esp

def objective(trial):
    scaler_name = trial.suggest_categorical("scaler", ["Standard", "Robust", "MinMax"])
    scaler = StandardScaler() if scaler_name == "Standard" else RobustScaler() if scaler_name == "Robust" else MinMaxScaler()
        
    k_features = trial.suggest_int("k_features", 5, 20)
    selector_name = trial.suggest_categorical("selector", ["KBest", "ExtraTrees"])
    selector = SelectKBest(f_classif, k=k_features) if selector_name == "KBest" else SelectFromModel(ExtraTreesClassifier(n_estimators=50, random_state=42), max_features=k_features)

    model_name = trial.suggest_categorical("model", ["LogReg", "SVM_Linear", "SVM_RBF", "GradientBoosting"])
    if model_name == "LogReg":
        modelo = LogisticRegression(C=trial.suggest_float("C_lr", 0.01, 10.0, log=True), class_weight='balanced', random_state=42)
    elif model_name == "SVM_Linear":
        modelo = SVC(kernel='linear', C=trial.suggest_float("C_svml", 0.01, 10.0, log=True), class_weight='balanced', random_state=42)
    elif model_name == "SVM_RBF":
        modelo = SVC(kernel='rbf', C=trial.suggest_float("C_svmr", 0.1, 20.0, log=True), class_weight='balanced', random_state=42)
    else:
        # Gradient Boosting controlado para evitar overfitting
        modelo = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=trial.suggest_float("lr_gb", 0.01, 0.2, log=True), random_state=42)

    pipeline = Pipeline([('scaler', scaler), ('selector', selector), ('classifier', modelo)])
    
    # 3-Fold para o Optuna não decorar os dados
    try:
        y_pred_cv = cross_val_predict(pipeline, X, y, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42))
        return calcular_score_clinico(y, y_pred_cv)[0]
    except: return 0.0

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)
best = study.best_params

# ---------------------------------------------------------
# 4. VALIDAÇÃO CRUZADA ESTREITA (LOOCV SEM VAZAMENTO)
# ---------------------------------------------------------
scaler = StandardScaler() if best['scaler'] == "Standard" else RobustScaler() if best['scaler'] == "Robust" else MinMaxScaler()
selector = SelectKBest(f_classif, k=best['k_features']) if best['selector'] == "KBest" else SelectFromModel(ExtraTreesClassifier(n_estimators=50, random_state=42), max_features=best['k_features'])

if best['model'] == "LogReg": best_model = LogisticRegression(C=best['C_lr'], class_weight='balanced')
elif best['model'] == "SVM_Linear": best_model = SVC(kernel='linear', C=best['C_svml'], class_weight='balanced')
elif best['model'] == "SVM_RBF": best_model = SVC(kernel='rbf', C=best['C_svmr'], class_weight='balanced')
else: best_model = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=best['lr_gb'], random_state=42)

pipe_final = Pipeline([('scaler', scaler), ('selector', selector), ('classifier', best_model)])

previsoes = []
for train_idx, test_idx in LeaveOneOut().split(X):
    pipe_final.fit(X.iloc[train_idx], y.iloc[train_idx])
    previsoes.append(pipe_final.predict(X.iloc[test_idx])[0])
    
_, acc_42, sens_42, esp_42 = calcular_score_clinico(y, previsoes)

clear_output(wait=True)
print(f"{'OPTUNA':^90}")
print(f"Arquitetura: {best['model']} | Scaler: {best['scaler']} | Seletor: {best['selector']} (K={best['k_features']})")
print("-" * 90)
print(f"RESULTADO CLÍNICO (LOOCV - sem vazamento):")
print(f"   Acurácia: {acc_42*100:.1f}% | Sensibilidade (TEA): {sens_42*100:.1f}% | Especificidade (Ctrl): {esp_42*100:.1f}%")
print("="*90)

pipe_final.fit(X, y)
features_ativas = X.columns[pipe_final.named_steps['selector'].get_support()].tolist()
print("\n VARIÁVEIS MAIS CLÍNICAS SEGUNDO O OPTUNA:")
for i, f in enumerate(features_ativas, 1): print(f"   {i}. {f}")

                                          OPTUNA                                          
Arquitetura: LogReg | Scaler: Robust | Seletor: KBest (K=20)
------------------------------------------------------------------------------------------
RESULTADO CLÍNICO (LOOCV - sem vazamento):
   Acurácia: 78.9% | Sensibilidade (TEA): 82.4% | Especificidade (Ctrl): 76.2%

 VARIÁVEIS MAIS CLÍNICAS SEGUNDO O OPTUNA:
   1. Perc_Boca_Raiva
   2. Perc_Nariz_Raiva
   3. Pupila_Media_Raiva
   4. Pupila_Std_Raiva
   5. GazeX_Std_Raiva
   6. GazeY_Std_Raiva
   7. Pupila_Media_Feliz
   8. GazeX_Std_Feliz
   9. GazeY_Std_Feliz
   10. Perc_Olhos_Neutro
   11. Perc_Nariz_Neutro
   12. Pupila_Media_Neutro
   13. Pupila_Std_Neutro
   14. GazeX_Std_Neutro
   15. GazeY_Std_Neutro
   16. Delta_NeutroRaiva_Perc_Boca
   17. Delta_FelizRaiva_Pupila_Media
   18. Trans_Olhos_Nariz
   19. Trans_Nariz_Boca
   20. Trans_Boca_Olhos


In [4]:
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')

print("="*80)
print("ROTA 1: VERSÃO DEMONSTRATIVA DE ARMADILHA ESTATÍSTICA (DATA LEAKAGE)")
print("Objetivo: Encontrar a semente e o K que geram os 89% no Gradient Boosting.")
print("="*80)

CAMINHO_MATRIZ = '/workspaces/EyeTracking/data/processed/matriz_features_dinamicas.csv'
df = pd.read_csv(CAMINHO_MATRIZ)

y = (df['Grupo'] == 'TEA').astype(int)
X = df.drop(columns=['Paciente', 'Grupo'])

# O VAZAMENTO DE DADOS (Padronização feita na base inteira, não no fold)
X_scaled = StandardScaler().fit_transform(X)

melhor_acc_global = 0
melhor_semente_global = 0
melhor_k_global = 0

loo = LeaveOneOut()

print("Iniciando a varredura de Sementes e K-Features com dados vazados. Aguarde...")

# Varrendo K de 5 a 20 e Sementes de 1 a 100
for k in range(5, 21):
    # O VAZAMENTO GRAVE: O seletor "olha" para o Y de todos os pacientes antes do LOOCV
    seletor = SelectKBest(f_classif, k=k)
    X_vazado = seletor.fit_transform(X_scaled, y)
    
    for seed in range(1, 101):
        clf = GradientBoostingClassifier(random_state=seed)
        previsoes = []
        
        for train_idx, test_idx in loo.split(X_vazado):
            # O modelo treina numa matriz que já sabe quais features são boas para o test_idx
            clf.fit(X_vazado[train_idx], y.iloc[train_idx])
            previsoes.append(clf.predict(X_vazado[test_idx])[0])
            
        acc = accuracy_score(y, previsoes)
        
        if acc > melhor_acc_global:
            melhor_acc_global = acc
            melhor_semente_global = seed
            melhor_k_global = k

clear_output(wait=True)
print("="*80)
print(f"{'RESULTADO DA AUDITORIA DE VAZAMENTO':^80}")
print("="*80)
print(f"Acurácia Máxima Atingida: {melhor_acc_global * 100:.1f}%")
print(f"Configuração do Erro: Gradient Boosting | Semente = {melhor_semente_global} | K-Features = {melhor_k_global}")
print("="*80)
print("Esta é a prova matemática do vazamento. Salve este número para a reunião.")

ROTA 1: VERSÃO DEMONSTRATIVA DE ARMADILHA ESTATÍSTICA (DATA LEAKAGE)
Objetivo: Encontrar a semente e o K que geram os 89% no Gradient Boosting.
Iniciando a varredura de Sementes e K-Features com dados vazados. Aguarde...


KeyboardInterrupt: 

In [5]:
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')

print("="*80)
print("ROTA 1 (VERSÃO EXPRESSA): DEMONSTRAÇÃO DE VAZAMENTO DE DADOS")
print("Objetivo: Encontrar a acurácia inflada com Gradient Boosting em < 2 minutos.")
print("="*80)

CAMINHO_MATRIZ = '/workspaces/EyeTracking/data/processed/matriz_features_dinamicas.csv'
df = pd.read_csv(CAMINHO_MATRIZ)

y = (df['Grupo'] == 'TEA').astype(int)
X = df.drop(columns=['Paciente', 'Grupo'])

# Padronização na base inteira
X_scaled = StandardScaler().fit_transform(X)

# Fixamos K=15 para evitar o loop gigantesco
k_fixo = 15
seletor = SelectKBest(f_classif, k=k_fixo)

# O VAZAMENTO CLÁSSICO: O seletor escolhe as features olhando para a resposta (Y) de TODOS os pacientes
X_vazado = seletor.fit_transform(X_scaled, y)

melhor_acc_global = 0
melhor_semente_global = 0

loo = LeaveOneOut()

print(f"Varrendo 50 sementes no LOOCV com matriz vazada... Aguarde.")

for seed in range(1, 51):
    # Limitamos n_estimators a 50 para processamento super rápido
    clf = GradientBoostingClassifier(n_estimators=50, random_state=seed)
    previsoes = []
    
    for train_idx, test_idx in loo.split(X_vazado):
        clf.fit(X_vazado[train_idx], y.iloc[train_idx])
        previsoes.append(clf.predict(X_vazado[test_idx])[0])
        
    acc = accuracy_score(y, previsoes)
    
    if acc > melhor_acc_global:
        melhor_acc_global = acc
        melhor_semente_global = seed

clear_output(wait=True)
print("="*80)
print(f"{'RESULTADO DA AUDITORIA DE VAZAMENTO':^80}")
print("="*80)
print(f"Acurácia Máxima Atingida: {melhor_acc_global * 100:.1f}%")
print(f"Configuração: Gradient Boosting | Semente = {melhor_semente_global} | K-Features = {k_fixo}")
print("="*80)
print("Se o valor estiver próximo a 89%, a origem do resultado está comprovada.")

                      RESULTADO DA AUDITORIA DE VAZAMENTO                       
Acurácia Máxima Atingida: 76.3%
Configuração: Gradient Boosting | Semente = 39 | K-Features = 15
Se o valor estiver próximo a 89%, a origem do resultado está comprovada.


In [6]:
import os
import glob
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import warnings
from IPython.display import clear_output

warnings.filterwarnings('ignore')

print("="*80)
print("ROTA 2: ASSINATURA CLÍNICA (MARKOV SCANPATH COM AOIS DINÂMICAS)")
print("Objetivo: Otimização rigorosa e sem vazamento focada na Raiva (Humano)")
print("="*80)

print("Lendo aoi.csv e construindo matrizes probabilísticas...")
caminho_aoi = '/workspaces/EyeTracking/data/aoi/aoi.csv'
df_aoi = pd.read_csv(caminho_aoi)
dicionario_aoi = {}

def extrair_xy(texto):
    if pd.isna(texto) or not isinstance(texto, str): return None
    match = re.search(r'\((\d+),\s*(\d+)\)', texto)
    return (int(match.group(1)), int(match.group(2))) if match else None

for col in df_aoi.columns[1:]:
    try:
        f_id = int(col)
        dicionario_aoi[f_id] = {}
        for i, regiao in enumerate(['Olho_D', 'Olho_E', 'Nariz', 'Boca']):
            p1, p2 = extrair_xy(df_aoi.loc[i*2, col]), extrair_xy(df_aoi.loc[(i*2)+1, col])
            if p1 and p2:
                dicionario_aoi[f_id][regiao] = (min(p1[0], p2[0]), min(p1[1], p2[1]), max(p1[0], p2[0]), max(p1[1], p2[1]))
    except: pass

def dentro_caixa(x, y, caixa):
    if pd.isna(x) or pd.isna(y) or not caixa: return False
    return (caixa[0] <= x <= caixa[2]) and (caixa[1] <= y <= caixa[3])

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_markov = []
estados = ['Olhos', 'Nariz', 'Boca', 'Fora']

for caminho in arquivos:
    nome = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome.upper() else "CONTROLE"
    try:
        df = pd.read_csv(caminho)
        df_raiva = df[(df['Fase_Estimulo'] == 'Exposicao_Face') & (df['Tipo_Estimulo'] == 'Humano') & (df['Emocao'] == 'Raiva')].copy()
        if len(df_raiva) == 0: continue
            
        def classificar_estado(row):
            fid = row['Face_ID']
            if fid not in dicionario_aoi: return 'Fora'
            x, y = row['Gaze_X'], row['Gaze_Y']
            if dentro_caixa(x, y, dicionario_aoi[fid].get('Olho_D')) or dentro_caixa(x, y, dicionario_aoi[fid].get('Olho_E')): return 'Olhos'
            if dentro_caixa(x, y, dicionario_aoi[fid].get('Boca')): return 'Boca'
            if dentro_caixa(x, y, dicionario_aoi[fid].get('Nariz')): return 'Nariz'
            return 'Fora'

        df_raiva['Estado_Atual'] = df_raiva.apply(classificar_estado, axis=1)
        
        mudancas = df_raiva[df_raiva['Estado_Atual'] != df_raiva['Estado_Atual'].shift(1)]
        sequencia = mudancas['Estado_Atual'].tolist()
        
        features_paciente = {'Paciente': nome, 'Grupo': grupo}
        for o in estados:
            for d in estados: features_paciente[f"Trans_{o}_{d}"] = 0.0
                
        transicoes = pd.DataFrame({'Origem': sequencia[:-1], 'Destino': sequencia[1:]})
        if len(transicoes) > 0:
            matriz_prob = pd.crosstab(transicoes['Origem'], transicoes['Destino'], normalize='index')
            for o in matriz_prob.index:
                for d in matriz_prob.columns: features_paciente[f"Trans_{o}_{d}"] = matriz_prob.loc[o, d] * 100
                    
        lista_markov.append(features_paciente)
    except: pass

df_markov = pd.DataFrame(lista_markov)
y = (df_markov['Grupo'] == 'TEA').astype(int)
X = df_markov.drop(columns=['Paciente', 'Grupo'])
X = X.loc[:, (X != 0).any(axis=0)] 

modelos = {
    "SVM Linear": SVC(kernel='linear', class_weight='balanced', random_state=42),
    "Regressão Logística": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
}

loo = LeaveOneOut()
resultados = []

print("Procurando a assinatura neurofisiológica limpa...")

for nome_modelo, algoritmo in modelos.items():
    melhor_acc, melhor_k, melhor_sens, melhor_esp = 0, 3, 0, 0
    
    for k in range(3, min(12, X.shape[1])):
        pipeline = Pipeline([('scaler', StandardScaler()), ('seletor', SelectKBest(f_classif, k=k)), ('classificador', algoritmo)])
        previsoes = []
        y_verdadeiro = []
        
        for train_index, test_index in loo.split(X):
            pipeline.fit(X.iloc[train_index], y.iloc[train_index])
            previsoes.append(pipeline.predict(X.iloc[test_index])[0])
            y_verdadeiro.append(y.iloc[test_index].values[0])
            
        acc = accuracy_score(y_verdadeiro, previsoes)
        if acc > melhor_acc:
            melhor_acc, melhor_k = acc, k
            cm = confusion_matrix(y_verdadeiro, previsoes)
            vp, fn, vn, fp = cm[1][1], cm[1][0], cm[0][0], cm[0][1]
            melhor_sens = (vp / (vp + fn)) * 100 if (vp + fn) > 0 else 0
            melhor_esp = (vn / (vn + fp)) * 100 if (vn + fp) > 0 else 0

    resultados.append({'Modelo': nome_modelo, 'Qtd Rotas (K)': melhor_k, 'Acurácia': melhor_acc * 100, 'Sensib (TEA)': melhor_sens, 'Especif (Ctrl)': melhor_esp})

clear_output(wait=True)
df_res = pd.DataFrame(resultados).sort_values(by='Sensib (TEA)', ascending=False).reset_index(drop=True)

print("="*90)
print(f"{'VEREDITO DA MATRIZ DE MARKOV: ASSINATURA CLÍNICA (NOVAS AOIs)':^90}")
print("="*90)
print(df_res.to_string(index=False, formatters={'Acurácia': '{:,.1f}%'.format, 'Sensib (TEA)': '{:,.1f}%'.format, 'Especif (Ctrl)': '{:,.1f}%'.format}))
print("="*90)

              VEREDITO DA MATRIZ DE MARKOV: ASSINATURA CLÍNICA (NOVAS AOIs)               
             Modelo  Qtd Rotas (K) Acurácia Sensib (TEA) Especif (Ctrl)
         SVM Linear              7    73.7%        64.7%          81.0%
Regressão Logística              6    65.8%        58.8%          71.4%
